# Qwen3-4B-Instruct-2507: QLoRA + FSDP

In [ ]:
!pip -q install -U "transformers>=4.55.0" "accelerate>=1.9.0" "peft>=0.16.0" "bitsandbytes>=0.46.0" "datasets>=3.6.0"

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("bf16 supported:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

!nvidia-smi

In [ ]:
%%writefile train_qwen3_avito_qlora_fsdp.py
import json
import os
import re
from pathlib import Path
from transformers.trainer_utils import get_last_checkpoint
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
OUTPUT_DIR = "/kaggle/working/qwen3-4b-avito-qlora" if Path("/kaggle/working").exists() else "qwen3-4b-avito-qlora"

SEED = 42
MAX_SEQ_LENGTH = 768
MAX_TARGET_CHARS = 360
VAL_SIZE = 0.05

NUM_EPOCHS = 2
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

SYSTEM_PROMPT = (
    "Ты русскоязычный помощник для объявлений. "
    "Пиши короткие живые описания как обычный продавец. "
    "Сохраняй бренды, модели, числа, размеры и редкие слова без изменений."
)

def find_data_path() -> Path:
    names = ("dataset_filtered_with_params.csv", "dataset_filtered.csv")
    for name in names:
        path = Path(name)
        if path.exists():
            return path
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for name in names:
            matches = sorted(kaggle_input.rglob(name))
            if matches:
                return matches[0]
    raise FileNotFoundError("Не найден датасет CSV.")

def clean_text(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).replace("\r", " ").replace("\n", " ")
    return re.sub(r"\s+", " ", text).strip()

def format_params(raw, max_items: int = 16) -> str:
    if pd.isna(raw) or str(raw).strip() == "":
        return ""
    try:
        data = json.loads(raw) if isinstance(raw, str) else raw
    except Exception:
        return clean_text(raw)[:500]
    if not isinstance(data, dict):
        return clean_text(raw)[:500]
    parts = []
    for key, value in data.items():
        if value is None or clean_text(value) == "":
            continue
        key = clean_text(str(key).replace("_", " "))
        if key.lower() in {"title", "заголовок"}:
            continue
        parts.append(f"{key}: {clean_text(value)}")
        if len(parts) >= max_items:
            break
    return "; ".join(parts)

def make_target(description: str) -> str:
    text = clean_text(description)
    sentences = re.split(r"(?<=[.!?])\s+", text)
    target = " ".join(sentences[:2]).strip() or text
    if len(target) <= MAX_TARGET_CHARS:
        return target
    cut = target[:MAX_TARGET_CHARS]
    last_stop = max(cut.rfind("."), cut.rfind("!"), cut.rfind("?"))
    if last_stop >= 80:
        return cut[: last_stop + 1]
    return cut.rstrip(" ,;:-") + "."

def build_user_prompt(example) -> str:
    lines = [
        "Напиши короткое живое описание объявления по заголовку и параметрам.",
        "Не добавляй точные факты, которых нет во входе.",
        f"Заголовок: {example['title']}",
    ]
    if example.get("category_name"):
        lines.append(f"Категория: {example['category_name']}\")")
    microcat = example.get("microcat_name", "")
    if microcat and microcat != "Неизвестная микрокатегория":
        lines.append(f"Микрокатегория: {microcat}")
    if example.get("params_text"):
        lines.append(f"Параметры: {example['params_text']}")
    lines.append("Верни только текст описания.")
    return "\n".join(lines)

def load_dataframe() -> pd.DataFrame:
    data_path = find_data_path()
    df = pd.read_csv(data_path)
    df["title"] = df["title"].map(clean_text)
    df["description"] = df["description"].map(clean_text)
    for column in ("category_name", "microcat_name", "params"):
        if column not in df.columns:
            df[column] = ""
        df[column] = df[column].map(clean_text)
    df["params_text"] = df["params"].map(format_params)
    df["target"] = df["description"].map(make_target)
    df = df[df["title"].str.len().between(3, 220) & df["target"].str.len().between(30, MAX_TARGET_CHARS + 20)].copy()
    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return df

class CausalLMCollator:
    def __init__(self, pad_token_id: int):
        self.pad_token_id = pad_token_id
    def __call__(self, features):
        max_len = max(len(item["input_ids"]) for item in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for item in features:
            pad_len = max_len - len(item["input_ids"])
            batch["input_ids"].append(item["input_ids"] + [self.pad_token_id] * pad_len)
            batch["attention_mask"].append(item["attention_mask"] + [0] * pad_len)
            batch["labels"].append(item["labels"] + [-100] * pad_len)
        return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}

def main():
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)

    compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    device_map = {"": local_rank} if torch.cuda.is_available() else None

    df = load_dataframe()
    val_rows = max(1, int(len(df) * VAL_SIZE))
    eval_df = df.iloc[:val_rows].reset_index(drop=True)
    train_df = df.iloc[val_rows:].reset_index(drop=True)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    def tokenize_row(example):
        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(example)},
        ]
        full_messages = prompt_messages + [{"role": "assistant", "content": example["target"]}]
        prompt_out = tokenizer.apply_chat_template(prompt_messages, tokenize=True, add_generation_prompt=True)
        if hasattr(prompt_out, "ids"): prompt_ids = prompt_out.ids
        elif isinstance(prompt_out, dict) or hasattr(prompt_out, "input_ids"): prompt_ids = prompt_out["input_ids"]
        else: prompt_ids = prompt_out

        full_out = tokenizer.apply_chat_template(full_messages, tokenize=True, add_generation_prompt=False)
        if hasattr(full_out, "ids"): input_ids = full_out.ids
        elif isinstance(full_out, dict) or hasattr(full_out, "input_ids"): input_ids = full_out["input_ids"]
        else: input_ids = full_out

        input_ids = list(input_ids)[:MAX_SEQ_LENGTH]
        prompt_ids = list(prompt_ids)
        prompt_len = min(len(prompt_ids), len(input_ids))
        labels = input_ids.copy()
        labels[:prompt_len] = [-100] * prompt_len
        return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}

    train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
    eval_dataset = Dataset.from_pandas(eval_df, preserve_index=False)
    train_dataset = train_dataset.map(tokenize_row, remove_columns=train_dataset.column_names)
    eval_dataset = eval_dataset.map(tokenize_row, remove_columns=eval_dataset.column_names)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_quant_storage=compute_dtype,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, torch_dtype=compute_dtype,
        device_map=device_map, attn_implementation="sdpa", trust_remote_code=False,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", task_type="CAUSAL_LM", target_modules="all-linear",
    ))

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(compute_dtype)
    
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR, seed=SEED, num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE, per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, learning_rate=LEARNING_RATE,
        warmup_steps=10, lr_scheduler_type="cosine", optim="adamw_torch", max_grad_norm=0.3,
        logging_steps=10, 
        save_steps=20,
        save_total_limit=2, eval_strategy="steps", eval_steps=100,
        fp16=compute_dtype == torch.float16, bf16=compute_dtype == torch.bfloat16,
        report_to="none", remove_unused_columns=False, dataloader_num_workers=2,
    )

    trainer = Trainer(
        model=model, args=training_args, train_dataset=train_dataset,
        eval_dataset=eval_dataset, data_collator=CausalLMCollator(tokenizer.pad_token_id),
    )

    last_checkpoint = None
    if os.path.isdir(OUTPUT_DIR):
        last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

    try:
        trainer.train(resume_from_checkpoint=last_checkpoint)
    
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    if trainer.is_world_process_zero():
        import subprocess
        subprocess.run(f"cd /kaggle/working && zip -qr qwen3-4b-avito-qlora.zip qwen3-4b-avito-qlora", shell=True)

if __name__ == "__main__":
    main()

In [ ]:
NUM_PROCESSES = 2

!accelerate launch --multi_gpu --num_processes {NUM_PROCESSES} train_qwen3_avito_qlora_fsdp.py

In [ ]:
from pathlib import Path

output_dir = Path("/kaggle/working/qwen3-4b-avito-qlora")
if output_dir.exists():
    !cd /kaggle/working && zip -qr qwen3-4b-avito-qlora.zip qwen3-4b-avito-qlora
    print("Adapter:", output_dir)
    print("Archive:", "/kaggle/working/qwen3-4b-avito-qlora.zip")
else:
    print("Output dir not found yet:", output_dir)